# EXP031: 選択頻度上位10項目の重複分析

Bank 1について、各stepで5,000人に選ばれた頻度が高い項目を手法ごとに上位10個抽出し、固定epsilon版DQNの各gammaが通常MFIとOracle MFIのどちらに近い項目集合を選んでいるか比較します。

- 順位は選択人数の降順、同数の場合はitemIDの昇順で決定します。
- あるstepで選択されたユニーク項目が10個未満の場合は、存在する項目だけを使用します。
- 生の重複項目数と、集合サイズを補正するJaccard類似度を出力します。
- `Jaccard(DQN, Oracle MFI) - Jaccard(DQN, MFI)` が正ならOracle MFIに近く、負なら通常MFIに近いと判定します。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_project_root():
    candidates = []
    if "__file__" in globals():
        script_dir = Path(__file__).resolve().parent
        candidates.extend([script_dir, *script_dir.parents])

    cwd = Path.cwd().resolve()
    candidates.extend(
        [
            cwd,
            *cwd.parents,
            cwd / "Grad_Research",
            cwd
            / "Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History",
            Path("/content/Grad_Research"),
            Path(
                "/content/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History"
            ),
            Path("/content/drive/MyDrive/Grad_Research"),
            Path(
                "/content/drive/MyDrive/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History"
            ),
        ]
    )

    for root in candidates:
        if (root / "data").is_dir() and (root / "EXP031").is_dir():
            return root

    raise FileNotFoundError("Could not find the project root.")


ROOT = find_project_root()
RESULTS_DIR = ROOT / "EXP031" / "results"
BANK_TYPE = "uncor"
BANK_ID = 1
TOP_N = 10
DQN_GAMMAS = (0, 0.1, 0.3, 0.6, 0.9)
KEY_STEPS = (1, 5, 10, 20, 30, 40)
KEY_COLUMNS = ["userID", "step"]

print(f"Project root: {ROOT}")
print(f"Results dir : {RESULTS_DIR}")
print(f"Bank        : {BANK_TYPE} {BANK_ID}")
print(f"Top N       : {TOP_N}")
print(f"DQN gammas  : {DQN_GAMMAS}")

In [ ]:
def gamma_token(gamma):
    return str(gamma)


def read_selection_records(path):
    if not path.is_file():
        raise FileNotFoundError(f"Missing records file: {path}")

    frame = pd.read_csv(path, usecols=["userID", "step", "itemID"])
    for column in ["userID", "step", "itemID"]:
        frame[column] = pd.to_numeric(frame[column], errors="raise")
        if not np.isfinite(frame[column]).all():
            raise ValueError(f"{path.name}: {column} contains non-finite values.")
        if not np.allclose(frame[column], np.round(frame[column])):
            raise ValueError(f"{path.name}: {column} must contain integer values.")
        frame[column] = frame[column].astype(np.int64)

    if frame.duplicated(KEY_COLUMNS).any():
        raise ValueError(f"{path.name} has duplicate userID-step rows.")

    return frame.sort_values(KEY_COLUMNS).reset_index(drop=True)


def assert_same_keys(reference, candidate, label):
    if not reference[KEY_COLUMNS].equals(candidate[KEY_COLUMNS]):
        raise ValueError(
            f"{label} does not have the same userID-step rows as Oracle MFI."
        )


def top_items_by_step(frame, method, top_n):
    counts = (
        frame.groupby(["step", "itemID"], as_index=False)
        .size()
        .rename(columns={"size": "selection_count"})
        .sort_values(
            ["step", "selection_count", "itemID"],
            ascending=[True, False, True],
        )
    )
    counts["rank"] = counts.groupby("step").cumcount() + 1
    totals = frame.groupby("step").size().rename("n_examinees")
    counts = counts.join(totals, on="step")
    counts["selection_rate"] = counts["selection_count"] / counts["n_examinees"]
    counts["method"] = method
    return counts.loc[
        counts["rank"] <= top_n,
        [
            "step",
            "method",
            "rank",
            "itemID",
            "selection_count",
            "selection_rate",
            "n_examinees",
        ],
    ].reset_index(drop=True)


def jaccard_similarity(left, right):
    union = left | right
    return len(left & right) / len(union) if union else 1.0

In [ ]:
oracle_path = RESULTS_DIR / (f"records_{BANK_TYPE}_{BANK_ID}_MFI_oracle_MLE_python.csv")
mfi_path = RESULTS_DIR / f"records_{BANK_TYPE}_{BANK_ID}_MFI_MLE_python.csv"
dqn_paths = {
    gamma: RESULTS_DIR
    / f"records_{BANK_TYPE}_{BANK_ID}_DQN_MLE_gamma_{gamma_token(gamma)}.csv"
    for gamma in DQN_GAMMAS
}

oracle_records = read_selection_records(oracle_path)
mfi_records = read_selection_records(mfi_path)
dqn_records = {gamma: read_selection_records(path) for gamma, path in dqn_paths.items()}

assert_same_keys(oracle_records, mfi_records, "MFI")
for gamma, frame in dqn_records.items():
    assert_same_keys(oracle_records, frame, f"DQN gamma={gamma}")

print(f"Examinees   : {oracle_records['userID'].nunique():,}")
print(f"Test steps  : {oracle_records['step'].nunique():,}")
print(f"Record rows : {len(oracle_records):,}")
print(f"Oracle MFI  : {oracle_path.name}")
print(f"MFI         : {mfi_path.name}")
for gamma, path in dqn_paths.items():
    print(f"DQN gamma={gamma}: {path.name}")

In [ ]:
method_records = {
    "Oracle MFI": oracle_records,
    "MFI": mfi_records,
}
method_records.update(
    {f"DQN (gamma={gamma})": frame for gamma, frame in dqn_records.items()}
)
METHOD_ORDER = list(method_records)

top_item_frames = [
    top_items_by_step(frame, method, TOP_N) for method, frame in method_records.items()
]
top_items = pd.concat(top_item_frames, ignore_index=True)
method_rank = {method: rank for rank, method in enumerate(METHOD_ORDER)}
top_items["method_rank"] = top_items["method"].map(method_rank)
top_items = (
    top_items.sort_values(["step", "method_rank", "rank"])
    .drop(columns="method_rank")
    .reset_index(drop=True)
)

top_items_path = (
    RESULTS_DIR / f"top{TOP_N}_selected_items_by_step_{BANK_TYPE}_{BANK_ID}.csv"
)
top_items.to_csv(top_items_path, index=False)
print(f"Saved top-item table: {top_items_path}")
display(top_items.head(TOP_N * len(METHOD_ORDER)))

In [ ]:
top_item_sets = {
    (method, int(step)): set(group["itemID"].astype(int))
    for (method, step), group in top_items.groupby(["method", "step"])
}
steps = sorted(oracle_records["step"].unique())
overlap_rows = []

for step in steps:
    oracle_set = top_item_sets[("Oracle MFI", step)]
    mfi_set = top_item_sets[("MFI", step)]
    for gamma in DQN_GAMMAS:
        dqn_method = f"DQN (gamma={gamma})"
        dqn_set = top_item_sets[(dqn_method, step)]
        overlap_with_mfi = dqn_set & mfi_set
        overlap_with_oracle = dqn_set & oracle_set
        jaccard_mfi = jaccard_similarity(dqn_set, mfi_set)
        jaccard_oracle = jaccard_similarity(dqn_set, oracle_set)
        jaccard_difference = jaccard_oracle - jaccard_mfi
        if jaccard_difference > 0:
            closer_to = "Oracle MFI"
        elif jaccard_difference < 0:
            closer_to = "MFI"
        else:
            closer_to = "Tie"

        overlap_rows.append(
            {
                "step": step,
                "gamma": gamma,
                "dqn_top_item_count": len(dqn_set),
                "mfi_top_item_count": len(mfi_set),
                "oracle_top_item_count": len(oracle_set),
                "overlap_count_with_mfi": len(overlap_with_mfi),
                "overlap_count_with_oracle": len(overlap_with_oracle),
                "overlap_items_with_mfi": "|".join(map(str, sorted(overlap_with_mfi))),
                "overlap_items_with_oracle": "|".join(
                    map(str, sorted(overlap_with_oracle))
                ),
                "jaccard_with_mfi": jaccard_mfi,
                "jaccard_with_oracle": jaccard_oracle,
                "jaccard_oracle_minus_mfi": jaccard_difference,
                "closer_to": closer_to,
            }
        )

overlap_summary = pd.DataFrame(overlap_rows).sort_values(["step", "gamma"])
overlap_path = RESULTS_DIR / f"dqn_top{TOP_N}_item_overlap_{BANK_TYPE}_{BANK_ID}.csv"
overlap_summary.to_csv(overlap_path, index=False)
print(f"Saved overlap summary: {overlap_path}")
display(overlap_summary.head(len(DQN_GAMMAS)))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6), constrained_layout=True)
for gamma in DQN_GAMMAS:
    curve = overlap_summary[overlap_summary["gamma"] == gamma]
    axes[0].plot(
        curve["step"],
        curve["jaccard_with_mfi"],
        marker="o",
        markersize=3,
        linewidth=1.8,
        label=f"gamma={gamma}",
    )
    axes[1].plot(
        curve["step"],
        curve["jaccard_with_oracle"],
        marker="o",
        markersize=3,
        linewidth=1.8,
        label=f"gamma={gamma}",
    )

axes[0].set_title(f"DQN top-{TOP_N} vs MFI top-{TOP_N}")
axes[1].set_title(f"DQN top-{TOP_N} vs Oracle MFI top-{TOP_N}")
for axis in axes:
    axis.set_xlabel("Step")
    axis.set_ylabel("Jaccard similarity")
    axis.set_xticks(np.arange(5, 41, 5))
    axis.set_ylim(-0.02, 1.02)
    axis.grid(alpha=0.3)
    axis.legend(fontsize=9)

fig.suptitle(
    f"Overlap of the top-{TOP_N} most frequently selected items\n"
    f"bank={BANK_TYPE} {BANK_ID}",
    fontsize=15,
)
overlap_figure_path = (
    RESULTS_DIR / f"dqn_top{TOP_N}_item_jaccard_{BANK_TYPE}_{BANK_ID}.png"
)
fig.savefig(overlap_figure_path, dpi=200, bbox_inches="tight")
print(f"Saved overlap figure: {overlap_figure_path}")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
for gamma in DQN_GAMMAS:
    curve = overlap_summary[overlap_summary["gamma"] == gamma]
    ax.plot(
        curve["step"],
        curve["jaccard_oracle_minus_mfi"],
        marker="o",
        markersize=3,
        linewidth=1.8,
        label=f"gamma={gamma}",
    )

ax.axhline(0.0, color="black", linewidth=1.2, linestyle="--")
ax.set_title(
    f"Which top-{TOP_N} item set is DQN closer to?\nPositive: Oracle MFI, negative: MFI"
)
ax.set_xlabel("Step")
ax.set_ylabel("Jaccard difference: Oracle MFI - MFI")
ax.set_xticks(np.arange(5, 41, 5))
ax.grid(alpha=0.3)
ax.legend(fontsize=9)
difference_figure_path = RESULTS_DIR / (
    f"dqn_top{TOP_N}_item_jaccard_oracle_minus_mfi_{BANK_TYPE}_{BANK_ID}.png"
)
fig.savefig(difference_figure_path, dpi=200, bbox_inches="tight")
print(f"Saved comparison figure: {difference_figure_path}")
plt.show()

In [ ]:
key_step_overlap = overlap_summary[overlap_summary["step"].isin(KEY_STEPS)].copy()
key_step_path = RESULTS_DIR / (
    f"dqn_top{TOP_N}_item_overlap_key_steps_{BANK_TYPE}_{BANK_ID}.csv"
)
key_step_overlap.to_csv(key_step_path, index=False)

print(f"Saved key-step table: {key_step_path}")
print("Jaccard difference: positive=Oracle MFI, negative=MFI")
display(
    key_step_overlap.pivot(
        index="gamma",
        columns="step",
        values="jaccard_oracle_minus_mfi",
    )
)
print("Closer reference counts across all 40 steps:")
display(overlap_summary.groupby(["gamma", "closer_to"]).size().unstack(fill_value=0))